# Module 2 : Les fondamentaux du machine learning

*Mardi 23 juin 2026, 9h-12h · Intervenant : Corentin Vande Kerckhove*

**CONSEIL :** Sauvegardez une copie de ce notebook dans votre Drive avant de commencer !

**Objectifs d'apprentissage**
- Comprendre ce qu'est une **feature** (et pourquoi il en faut)
- **Nettoyer** un dataset : valeurs manquantes, normalisation
- Comprendre pourquoi on sépare les données en **train** et **test**
- **Entraîner** un modèle et lire ce qu'il a appris
- Choisir une **métrique** qui ne ment pas (accuracy, precision, recall)
- *(en option)* diagnostiquer l'**overfitting** et passer à la **régression**

**Pré-requis :** Module 1 (notion de feature, de vectorisation).
**Paquets requis :** `scikit-learn`, `pandas`, `matplotlib`

## Le running example : Law School

Un dataset classique en sociologie quantitative du droit et en fairness ML : la *LSAC National Longitudinal Bar Passage Study* (Wightman, 1998). On suit environ 18 700 étudiant·es de droit aux États-Unis, depuis leur entrée en faculté jusqu'à leur passage du barreau (l'examen national pour exercer comme avocat·e).

Le parcours :

```
LSAT + UGPA → 1ère année → ... → Law School GPA → Bar Exam → Pass/Fail
```

On prédira **`y_pass_bar`** : a-t-on réussi le barreau (0/1) ? À partir des seules variables connues **à l'entrée** en faculté de droit (notes pré-droit, statut temps plein, revenu familial, tier de la faculté).

Le dataset est livré déjà nettoyé et déjà splitté (train/test) dans `data/law_school/`. Voir `data/law_school/codebook.md` pour le détail des colonnes.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j2/1_matin/img/m2/m2-opener.jpg" width="440" alt="Palais de justice">

*Running example : prédire la réussite à l'examen du barreau. Photo : palais de justice, cygnus921, CC BY 2.0.*

## Setup

In [ ]:
# Sur Colab : exécute cette cellule pour installer les paquets (déjà installés sur un poste local)
!pip install -q scikit-learn pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Un premier regard sur les données

Avant de modéliser quoi que ce soit, **ouvrons le dataset et regardons à quoi il ressemble** : combien de lignes, quelles colonnes, quelles valeurs.

In [ ]:
# Chargement des CSV pré-traités, déjà séparés en train et test
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/law_school"

df_train = pd.read_csv(f"{URL}/law_school_for_classification_train.csv")
df_test = pd.read_csv(f"{URL}/law_school_for_classification_test.csv")

print(f"Train : {len(df_train):,} lignes")
print(f"Test  : {len(df_test):,} lignes")
df_train.head()

### Que contient chaque colonne ?

Chaque ligne est un·e étudiant·e ; chaque colonne, une information mesurée sur cette personne.

| Colonne | Ce qu'elle mesure | Échelle |
| - | - | - |
| `x_lsat` | Score au test d'admission en fac de droit (LSAT) | ~11 à 48 |
| `x_ugpa` | Moyenne **avant** la fac de droit (GPA undergraduate) | 0 à 4 |
| `x_fulltime` | Inscrit·e à temps plein ? | 0 / 1 |
| `x_fam_inc` | Revenu familial déclaré | 1 (bas) à 5 (haut) |
| `x_tier` | Rang / prestige de la faculté de droit | 1 à 6 |
| `y_pass_bar` | A réussi l'examen du barreau ? | 0 / 1 |
| `z_white` | Indicateur démographique | 0 / 1 |

Les noms de colonnes commencent par `x_`, `y_` ou `z_` : ce n'est pas un hasard. On expliquera cette **convention de nommage** juste après, dans *La notion de feature*.

## Pourquoi deux jeux de données (train et test) ?

Un modèle ne doit pas seulement bien marcher sur les données qu'il a **déjà vues** : il doit **généraliser** à des données nouvelles. On sépare donc les données en deux :

- le **train** (75 %) : le modèle apprend dessus ;
- le **test** (25 %) : on mesure sa qualité dessus, sur des lignes qu'il n'a **jamais vues**.

Ici, la séparation est **déjà faite** (`_train.csv` / `_test.csv`) : cela garantit que personne ne triche en regardant le test pendant l'exploration.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j2/1_matin/img/m2/m2-split.png" width="560" alt="Schéma train/test split">

*On coupe le dataset en deux : on apprend sur le train, on évalue sur le test.*

## 2.1 La notion de feature

Une **feature** (variable explicative, prédicteur…) est une colonne dont le modèle se sert pour faire sa prédiction.

Au **module 1**, nos features étaient des **mots** : pour chaque mot-clé, on mettait `1` s'il apparaissait dans la phrase, `0` sinon. Mais une feature n'est **pas forcément un mot** : c'est **n'importe quel indicateur mesurable**. Ici, ce sont des notes, un revenu, un statut.

On peut maintenant lever le voile sur les **préfixes** des colonnes repérés plus haut :

| Préfixe | Rôle | Colonnes ici |
| :-: | - | - |
| `x_` | features prédictives | `x_lsat`, `x_ugpa`, `x_fulltime`, `x_fam_inc`, `x_tier` |
| `y_` | cible à prédire | `y_pass_bar` |
| `z_` | attribut sensible | `z_white` |

Les **`x_`** sont les **features** : ce sont elles que le modèle reçoit en entrée. La cible **`y_pass_bar`** vaut `1` si la personne a réussi l'examen du barreau, `0` sinon. Toutes les features sont des signaux disponibles **à l'entrée** en faculté (avant l'examen).

Le **`z_white`** est un **attribut sensible** qu'on **isole** : on ne l'utilise pas comme feature, et on s'en servira jeudi pour vérifier que le modèle n'est pas injuste.

On rassemble ces noms dans deux variables qu'on réutilisera partout :

In [ ]:
FEATURES = ["x_lsat", "x_ugpa", "x_fulltime", "x_fam_inc", "x_tier"]
TARGET = "y_pass_bar"

print("Features :", FEATURES)
print("Cible    :", TARGET)

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j2/1_matin/img/m2/m2-feature.png" width="560" alt="Schéma feature">

*Une feature est une caractéristique d'un individu transformée en colonne de chiffres.*

In [ ]:
# Aperçu des types de chaque colonne
df_train.dtypes

In [ ]:
# Proportion de réussite au barreau (la cible)
df_train["y_pass_bar"].value_counts(normalize=True).round(3)

### Hack Time

Affichez `df_train.describe()` et repérez la feature la plus **dispersée** (regardez les colonnes `min`, `max`, `std`).

In [ ]:
# Votre code ici

## 2.2 Nettoyage et préparation

Règle d'or : **garbage in, garbage out**. Un modèle entraîné sur des données pourries fait des prédictions pourries (souvent avec aplomb).

Comme souvent avec de vraies données, notre dataset n'est **pas parfait**. Deux chantiers très courants nous attendent :
- les **valeurs manquantes** (des trous dans le tableau) ;
- la **normalisation** des features (les ramener à une échelle commune).

On commence par **repérer** les trous, puis on les bouche, et enfin on normalise.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j2/1_matin/img/m2/m2-cleaning.png" width="560" alt="Schéma nettoyage">

*Avant de modéliser, on corrige les trous et on ramène les features à une échelle commune.*

In [ ]:
# Combien de valeurs manquantes dans chaque feature ? (train)
print("Valeurs manquantes par feature (train) :")
print(df_train[FEATURES].isna().sum())

**(a) Boucher les trous.** On remplace chaque valeur manquante par la **moyenne** de la feature, calculée sur le **train uniquement** (utiliser le test serait tricher).

In [ ]:
means = df_train[FEATURES].mean()
df_train_filled = df_train.fillna(means)
df_test_filled = df_test.fillna(means)

print("Valeurs manquantes restantes :", df_train_filled[FEATURES].isna().sum().sum())

**(b) Normaliser.** Les features n'ont pas la même échelle (un LSAT ~40 vs un revenu 1-5). On les ramène entre 0 et 1 en **divisant chaque feature par son maximum** (mesuré sur le train).

In [ ]:
maxima = df_train_filled[FEATURES].max()
X_train = df_train_filled[FEATURES] / maxima
X_test = df_test_filled[FEATURES] / maxima
y_train = df_train_filled[TARGET]
y_test = df_test_filled[TARGET]

X_train.describe().round(2)

### Hack Time

Vérifiez avec `X_test.describe()` que les features sont bien ramenées autour de 0-1 (le test peut légèrement dépasser 1 : pourquoi ?).

In [ ]:
# Votre code ici

## 2.3 Entraînement

On entraîne une **régression logistique** sur le train, puis on regarde ce qu'elle a retenu.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

print(f"Accuracy sur le test : {clf.score(X_test, y_test):.3f}")

### Que retient le modèle ?

Lire les coefficients relie le modèle à une interprétation humaine : une feature au coefficient **positif** pousse vers la réussite au barreau, **négatif** vers l'échec.

In [ ]:
pd.Series(clf.coef_[0], index=FEATURES).sort_values()

## 2.4 Métriques : accuracy, precision, recall

`accuracy` = la proportion de prédictions correctes. C'est intuitif, mais ça **ment** quand les classes sont déséquilibrées.

Ici, `y_pass_bar` vaut 1 (réussite) dans ~90 % des cas. Un modèle bidon qui dit **« tout le monde passe »** obtient déjà **90 % d'accuracy** sans rien apprendre.

Trois métriques se lisent dans la **matrice de confusion** :

|  | Prédit 1 (passe) | Prédit 0 (rate) |
|---|---|---|
| **Réel 1 (passe)** | TP (vrai positif) | FN (faux négatif) |
| **Réel 0 (rate)**  | FP (faux positif) | TN (vrai négatif) |

- **Accuracy** = (TP + TN) / total. « Combien de prédictions sont bonnes ? »
- **Precision** = TP / (TP + FP). « Quand on prédit *passe*, à quelle fréquence a-t-on raison ? »
- **Recall** = TP / (TP + FN). « Parmi celles et ceux qui passent vraiment, combien attrape-t-on ? »

Precision et recall sont en **tension**. Le **F1-score** est leur moyenne harmonique, pour les résumer en un chiffre.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j2/1_matin/img/m2/m2-confusion.png" width="560" alt="Schéma matrice de confusion">

*La matrice de confusion : d'où sortent accuracy, precision et recall.*

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = clf.predict(X_test)

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"F1        : {f1_score(y_test, y_pred):.3f}")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

### Hack Time

Construisez un modèle « bidon » qui prédit toujours `1` (réussite) et comparez son accuracy, sa precision et son recall à ceux du modèle. Que révèle son accuracy élevée ?

In [ ]:
# Votre code ici

## Récap module 2

Le fil du module : **feature → nettoyage → train/test → entraînement → métriques**.

L'idée à retenir : le framework est **le même** quel que soit le problème. On choisit des features, on nettoie, on sépare train/test, on entraîne, on mesure avec une métrique honnête.

→ Direction le **module 3** : on retourne au texte pour comprendre comment il devient des **tokens**, le carburant des LLMs.

Les deux sections ci-dessous sont **optionnelles** (à explorer si le temps le permet).

## (Optionnel) Et si la cible était continue ? La régression

Jusqu'ici on prédit du **binaire** (passe / rate). Souvent on veut prédire un **nombre** : un revenu, une note, une durée.

On rejoue le **même framework** en changeant trois choses :
1. **la cible** : `y_zgpa` (la moyenne en droit, continue) au lieu de `y_pass_bar` ;
2. **le modèle** : `LinearRegression` au lieu de `LogisticRegression` ;
3. **les métriques** : on mesure un **écart**, pas un taux de bonnes réponses :
   - **MAE** : erreur moyenne, dans l'unité de la cible ;
   - **RMSE** : pénalise davantage les grosses erreurs ;
   - **R²** : part de la variance expliquée (1 = parfait, 0 = pas mieux qu'une moyenne).

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m2/m2-regression.png" width="560" alt="Schéma régression">

*Même démarche qu'en classification, mais la cible devient un nombre continu au lieu d'une catégorie.*

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Mêmes individus, mêmes features, autre cible (continue)
df_train_reg = pd.read_csv(f"{URL}/law_school_for_regression_train.csv")
df_test_reg = pd.read_csv(f"{URL}/law_school_for_regression_test.csv")

reg = LinearRegression()
reg.fit(df_train_reg[FEATURES], df_train_reg["y_zgpa"])
y_pred_reg = reg.predict(df_test_reg[FEATURES])
y_true_reg = df_test_reg["y_zgpa"]

print(f"MAE  : {mean_absolute_error(y_true_reg, y_pred_reg):.3f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_true_reg, y_pred_reg)):.3f}")
print(f"R2   : {r2_score(y_true_reg, y_pred_reg):.3f}")
print("\n(Pas d'accuracy ni de precision : ces mesures n'existent que pour la classification.)")

### Hack Time

Faites un nuage de points `y_true_reg` (axe x) vs `y_pred_reg` (axe y). Si le modèle était parfait, tous les points seraient sur la diagonale. Qu'observe-t-on ?

In [ ]:
# Votre code ici

## (Optionnel) Overfitting

Le **piège classique du ML** : le modèle apprend ses données **par cœur**, performe à 100 % sur le train, et chute sur le test.

L'**arbre de décision** est la machine à fabriquer de l'overfitting : sans limite de profondeur, il mémorise les données d'entraînement. (Un arbre, c'est une cascade de questions « LSAT > 35 ? UGPA > 3.5 ? »… plus il est profond, plus il mémorise.)

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j2/1_matin/img/m2/m2-overfitting.png" width="560" alt="Schéma overfitting">

*Quand le modèle mémorise, l'erreur de test remonte alors que celle du train baisse.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Exactitude train vs test pour des arbres de plus en plus profonds
for depth in [1, 3, 5, 10, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_train, y_train)
    print(f"profondeur {str(depth):>4} : "
          f"train {tree.score(X_train, y_train):.3f} | test {tree.score(X_test, y_test):.3f}")

On voit l'exactitude **train** grimper vers 1.0 (l'arbre mémorise) pendant que l'exactitude **test** stagne, voire baisse : c'est l'overfitting.

### Hack Time

Tracez les deux courbes (train et test) en fonction de la profondeur, et repérez la profondeur où l'écart se creuse (le moment où le modèle commence à mémoriser).

In [ ]:
# Votre code ici